# AMPS I · Week 07 — Lab
## Regularity and convergence rates: measuring them correctly

| | |
|---|---|
| **Lecture sections** | Part A (Sobolev regularity, Céa, Aubin–Nitsche) · Part B (computing errors and rates) · Part C (corner singularities, graded meshes) · Part D (LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | see that the regularity of $u$, not the element, limits the rate · compare four ways of computing an error · measure the $H^1$ and $L^2$ rates on an L-shaped domain and compare with $\lambda=\pi/\omega$ · recover the optimal rate with a graded mesh |

**Reminder.** `%%fenicsx` cells run in a separate MPI process and are the only cells where `basix`, `ufl`, `dolfinx` are used.
Ordinary cells have NumPy and matplotlib only. Data passes through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · Regularity decides the rate  *(lecture §A2, §A5)*
Interpolate $u=r^\alpha$ ($r$ = distance to the corner $(0,0)$ of the unit square) into P$k$ and measure $\|u-\mathscr I_hu\|_{L^2}$ and
$|u-\mathscr I_hu|_{H^1}$. Since $r^\alpha\in H^s$ for $s<1+\alpha$, the prediction is $\min(k+1,\,1+\alpha)$ in $L^2$ and $\min(k,\,\alpha)$ in $H^1$.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

def errors(msh, e, q=10):
    """(L2 norm, H1 seminorm) of the UFL expression e, integrated directly with quadrature degree q, summed over ranks."""
    dx = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": q})
    s = lambda f: np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(f * dx)), op=MPI.SUM))
    return s(e * e), s(ufl.inner(ufl.grad(e), ufl.grad(e)))

def rates(E):
    """Rates between consecutive meshes that halve h."""
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2)

for alpha in (0.5, 1.5):
    for k in (1, 2):
        E = []
        for N in (8, 16, 32, 64):
            msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
            x = ufl.SpatialCoordinate(msh)
            u = ufl.sqrt(x[0] ** 2 + x[1] ** 2) ** alpha
            V = fem.functionspace(msh, ("Lagrange", k))
            Ih = fem.Function(V); Ih.interpolate(fem.Expression(u, V.element.interpolation_points))
            E.append(errors(msh, Ih - u, q=12))
        E = np.array(E)
        if MPI.COMM_WORLD.rank == 0:
            print(f"r^{alpha} P{k}: L2 rates {rates(E[:, 0])}  H1 rates {rates(E[:, 1])}   "
                  f"predicted {min(k + 1, 1 + alpha)} / {min(k, alpha)}")

> **✏️ Your turn (5 min).** **Predict first**, then run: $\alpha=2.5$ with P1, P2, P3. For which $k$ does the function limit the rate, and for which the element?

In [ ]:
%%fenicsx -np 2
# TODO: alpha = 2.5, P1-P3 (write your prediction in a comment first)

## 2 · Computing the error  *(lecture §B2)*
$-\Delta u=f$ on the unit square, $u=\sin\pi x\sin\pi y$, $u=0$ on $\partial\Omega$, P3, $N=8,\dots,128$. Four ways to compute "the $L^2$ error":

1. directly, $\int(u_h-u)^2$ with $u$ a UFL expression
2. expanded, $\int u^2-2\int uu_h+\int u_h^2$
3. against the interpolant of $u$ in the **same** space $V_h$
4. both $u_h$ and $u$ interpolated into P6, difference of the DOF vectors, then integrated

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

comm = MPI.COMM_WORLD
S = lambda f: comm.allreduce(fem.assemble_scalar(fem.form(f)), op=MPI.SUM)
k, Ns = 3, (8, 16, 32, 64, 128)
res = {"direct": [], "expanded": [], "same space": [], "P6 DOF-wise": []}
for N in Ns:
    msh = mesh.create_unit_square(comm, N, N)
    x = ufl.SpatialCoordinate(msh)
    u_ex = ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])
    V = fem.functionspace(msh, ("Lagrange", k))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    msh.topology.create_connectivity(1, 2)
    bc = fem.dirichletbc(0.0, fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology)), V)
    uh = LinearProblem(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, -ufl.div(ufl.grad(u_ex)) * v * ufl.dx, bcs=[bc],
                       petsc_options_prefix=f"err_{N}_", petsc_options={"ksp_type": "preonly", "pc_type": "lu"}).solve()
    res["direct"].append(np.sqrt(S((uh - u_ex) ** 2 * ufl.dx)))
    res["expanded"].append(np.sqrt(abs(S(u_ex ** 2 * ufl.dx) - 2 * S(u_ex * uh * ufl.dx) + S(uh ** 2 * ufl.dx))))
    Ih = fem.Function(V); Ih.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    res["same space"].append(np.sqrt(S((uh - Ih) ** 2 * ufl.dx)))
    W = fem.functionspace(msh, ("Lagrange", k + 3))
    uW = fem.Function(W); uW.interpolate(fem.Expression(u_ex, W.element.interpolation_points))
    eW = fem.Function(W); eW.interpolate(uh); eW.x.array[:] -= uW.x.array
    res["P6 DOF-wise"].append(np.sqrt(S(eW ** 2 * ufl.dx)))
if comm.rank == 0:
    for name, E in res.items():
        print(f"{name:12s} {[f'{e:.2e}' for e in E]}  rates {np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2)}")

> **✏️ Your turn (5 min).** Which methods agree? Explain the "expanded" numbers on the fine meshes (estimate the size of the round-off in
> $\int u^2-2\int uu_h+\int u_h^2$ when each term is about $0.25$). What does "same space" measure, and why is it not the error?

> *your answer*

## 3 · The L-shaped domain  *(lecture §A6–A7, §C1)*
$-\Delta u=0$ on $[-1,1]^2$ minus the quadrant $\{x>0,y<0\}$, $u=r^{2/3}\sin(2\theta/3)$ on the boundary. Interior angle $\omega=3\pi/2$ at the origin,
so $\lambda=\pi/\omega=2/3$. Prediction: $H^1$ rate $\lambda$, $L^2$ rate $2\lambda$ — for every degree. First a picture (one process).

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

def errors(msh, e, q=10):
    """(L2 norm, H1 seminorm) of the UFL expression e, integrated directly with quadrature degree q, summed over ranks."""
    dx = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": q})
    s = lambda f: np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(f * dx)), op=MPI.SUM))
    return s(e * e), s(ufl.inner(ufl.grad(e), ufl.grad(e)))

def rates(E):
    """Rates between consecutive meshes that halve h."""
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2)

def corner_domain(N, omega):
    """[-1,1]^2 (2N x 2N cells, h = 1/N) minus a wedge at the origin: interior angle omega = 5pi/4, 3pi/2 (L-shape) or 7pi/4."""
    big = mesh.create_rectangle(MPI.COMM_WORLD, [[-1, -1], [1, 1]], [2 * N, 2 * N], diagonal=mesh.DiagonalType.right)
    mid = mesh.compute_midpoints(big, 2, np.arange(big.topology.index_map(2).size_local, dtype=np.int32))
    if np.isclose(omega, 1.25 * np.pi):
        out = (mid[:, 1] < 0) & (mid[:, 1] < mid[:, 0])                      # y < 0 and below the line y = x removed
    elif np.isclose(omega, 1.5 * np.pi):
        out = (mid[:, 0] > 0) & (mid[:, 1] < 0)                              # quadrant x > 0, y < 0 removed
    else:
        out = (mid[:, 0] > 0) & (mid[:, 1] > 0) & (mid[:, 1] < mid[:, 0])    # wedge 0 < y < x removed
    return mesh.create_submesh(big, 2, np.where(~out)[0].astype(np.int32))[0]    # (mesh, entity map, vertex map, geom map)

def corner_exact(msh, omega):
    """u = r^lam sin(lam theta), lam = pi/omega, theta measured from one boundary ray of the corner (u = 0 on both rays)."""
    x = ufl.SpatialCoordinate(msh)
    r, th = ufl.sqrt(x[0] ** 2 + x[1] ** 2), ufl.atan2(x[1], x[0])       # atan2 in (-pi, pi]
    if omega < 1.6 * np.pi:
        th = ufl.conditional(ufl.lt(th, 0), th + 2 * ufl.pi, th)         # theta in [0, omega] for 5pi/4 and 3pi/2
    else:
        th = ufl.conditional(ufl.lt(th, np.pi / 4 - 1e-14), th + 2 * ufl.pi, th) - np.pi / 4   # [0, 7pi/4] from the ray y = x
    lam = np.pi / omega
    return r ** lam * ufl.sin(lam * th)

def corner_solve(msh, k, omega):
    """-Δu = 0 with u = exact on the boundary; returns (number of DOFs, L2 error, H1 error, u_h)."""
    V = fem.functionspace(msh, ("Lagrange", k))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    u_ex = corner_exact(msh, omega)
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    msh.topology.create_connectivity(1, 2)
    bc = fem.dirichletbc(g, fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology)))
    ndofs = V.dofmap.index_map.size_global
    uh = LinearProblem(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, fem.Constant(msh, 0.0) * v * ufl.dx, bcs=[bc],
                       petsc_options_prefix=f"corner_{k}_{ndofs}_", petsc_options={"ksp_type": "preonly", "pc_type": "lu"}).solve()
    return (ndofs, *errors(msh, uh - u_ex), uh)

msh = corner_domain(16, 1.5 * np.pi)
ndofs, l2, h1, uh = corner_solve(msh, 1, 1.5 * np.pi)
V = uh.function_space
print(f"{ndofs} DOFs, L2 error {l2:.2e}, H1 error {h1:.2e}")
np.savez("/tmp/fnx/lshape.npz", x=V.tabulate_dof_coordinates()[:, :2], cells=V.dofmap.list, u=uh.x.array)  # P1: DOF map, not geometry

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

d = np.load("/tmp/fnx/lshape.npz")
tri = mtri.Triangulation(d["x"][:, 0], d["x"][:, 1], d["cells"])
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
tc = axs[0].tripcolor(tri, d["u"], shading="gouraud"); fig.colorbar(tc, ax=axs[0]); axs[0].set_title("$u_h$ (P1, $h=1/16$)")
axs[0].set_aspect("equal"); axs[0].axis("off")
r = np.hypot(d["x"][:, 0], d["x"][:, 1]); sel = np.isclose(d["x"][:, 0], -d["x"][:, 1]) & (d["x"][:, 0] < 0)   # ray theta = 3pi/4
o = np.argsort(r[sel])
axs[1].plot(r[sel][o], d["u"][sel][o], "o-", ms=4, label="$u_h$ on $\\theta=3\\pi/4$")
rr = np.linspace(0, r[sel].max(), 200); axs[1].plot(rr, rr ** (2 / 3), "k:", label="$r^{2/3}$")
axs[1].set_xlabel("$r$"); axs[1].legend(); axs[1].grid(alpha=0.3); axs[1].set_title("infinite slope at the corner")
fig.tight_layout(); plt.show()

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

def errors(msh, e, q=10):
    """(L2 norm, H1 seminorm) of the UFL expression e, integrated directly with quadrature degree q, summed over ranks."""
    dx = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": q})
    s = lambda f: np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(f * dx)), op=MPI.SUM))
    return s(e * e), s(ufl.inner(ufl.grad(e), ufl.grad(e)))

def rates(E):
    """Rates between consecutive meshes that halve h."""
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2)

def corner_domain(N, omega):
    """[-1,1]^2 (2N x 2N cells, h = 1/N) minus a wedge at the origin: interior angle omega = 5pi/4, 3pi/2 (L-shape) or 7pi/4."""
    big = mesh.create_rectangle(MPI.COMM_WORLD, [[-1, -1], [1, 1]], [2 * N, 2 * N], diagonal=mesh.DiagonalType.right)
    mid = mesh.compute_midpoints(big, 2, np.arange(big.topology.index_map(2).size_local, dtype=np.int32))
    if np.isclose(omega, 1.25 * np.pi):
        out = (mid[:, 1] < 0) & (mid[:, 1] < mid[:, 0])                      # y < 0 and below the line y = x removed
    elif np.isclose(omega, 1.5 * np.pi):
        out = (mid[:, 0] > 0) & (mid[:, 1] < 0)                              # quadrant x > 0, y < 0 removed
    else:
        out = (mid[:, 0] > 0) & (mid[:, 1] > 0) & (mid[:, 1] < mid[:, 0])    # wedge 0 < y < x removed
    return mesh.create_submesh(big, 2, np.where(~out)[0].astype(np.int32))[0]    # (mesh, entity map, vertex map, geom map)

def corner_exact(msh, omega):
    """u = r^lam sin(lam theta), lam = pi/omega, theta measured from one boundary ray of the corner (u = 0 on both rays)."""
    x = ufl.SpatialCoordinate(msh)
    r, th = ufl.sqrt(x[0] ** 2 + x[1] ** 2), ufl.atan2(x[1], x[0])       # atan2 in (-pi, pi]
    if omega < 1.6 * np.pi:
        th = ufl.conditional(ufl.lt(th, 0), th + 2 * ufl.pi, th)         # theta in [0, omega] for 5pi/4 and 3pi/2
    else:
        th = ufl.conditional(ufl.lt(th, np.pi / 4 - 1e-14), th + 2 * ufl.pi, th) - np.pi / 4   # [0, 7pi/4] from the ray y = x
    lam = np.pi / omega
    return r ** lam * ufl.sin(lam * th)

def corner_solve(msh, k, omega):
    """-Δu = 0 with u = exact on the boundary; returns (number of DOFs, L2 error, H1 error, u_h)."""
    V = fem.functionspace(msh, ("Lagrange", k))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    u_ex = corner_exact(msh, omega)
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    msh.topology.create_connectivity(1, 2)
    bc = fem.dirichletbc(g, fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology)))
    ndofs = V.dofmap.index_map.size_global
    uh = LinearProblem(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, fem.Constant(msh, 0.0) * v * ufl.dx, bcs=[bc],
                       petsc_options_prefix=f"corner_{k}_{ndofs}_", petsc_options={"ksp_type": "preonly", "pc_type": "lu"}).solve()
    return (ndofs, *errors(msh, uh - u_ex), uh)

for k in (1, 2):
    R = [corner_solve(corner_domain(N, 1.5 * np.pi), k, 1.5 * np.pi)[:3] for N in (4, 8, 16, 32)]
    E = np.array([r_[1:] for r_ in R])
    if MPI.COMM_WORLD.rank == 0:
        print(f"P{k}: DOFs {[r_[0] for r_ in R]}  H1 {[f'{e:.2e}' for e in E[:, 1]]} rates {rates(E[:, 1])}"
              f"   L2 rates {rates(E[:, 0])}   predicted {2/3:.3f} / {4/3:.3f}")

> **✏️ Your turn (5 min).** Look at the P2 errors: P2 is still more **accurate** than P1 at the same $h$, but by a constant factor. How many DOFs does
> P2 need for $|u-u_h|_{H^1}\approx 3\times10^{-2}$, compared with P1? Is P2 worth it on this problem with uniform meshes?

> *your answer*

## 4 · Graded meshes  *(lecture §C3)*
Refine towards the corner until $h_K\le1.5\,H\,r_K^{1-\mu}$. With $\mu<\lambda/k$ the error should decay like $N_{\rm dof}^{-k/2}$ in $H^1$ — the rate of a smooth solution.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

def errors(msh, e, q=10):
    """(L2 norm, H1 seminorm) of the UFL expression e, integrated directly with quadrature degree q, summed over ranks."""
    dx = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": q})
    s = lambda f: np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(f * dx)), op=MPI.SUM))
    return s(e * e), s(ufl.inner(ufl.grad(e), ufl.grad(e)))

def rates(E):
    """Rates between consecutive meshes that halve h."""
    E = np.asarray(E, dtype=float)
    return np.round(np.log2(E[:-1] / E[1:]), 2)

def corner_domain(N, omega):
    """[-1,1]^2 (2N x 2N cells, h = 1/N) minus a wedge at the origin: interior angle omega = 5pi/4, 3pi/2 (L-shape) or 7pi/4."""
    big = mesh.create_rectangle(MPI.COMM_WORLD, [[-1, -1], [1, 1]], [2 * N, 2 * N], diagonal=mesh.DiagonalType.right)
    mid = mesh.compute_midpoints(big, 2, np.arange(big.topology.index_map(2).size_local, dtype=np.int32))
    if np.isclose(omega, 1.25 * np.pi):
        out = (mid[:, 1] < 0) & (mid[:, 1] < mid[:, 0])                      # y < 0 and below the line y = x removed
    elif np.isclose(omega, 1.5 * np.pi):
        out = (mid[:, 0] > 0) & (mid[:, 1] < 0)                              # quadrant x > 0, y < 0 removed
    else:
        out = (mid[:, 0] > 0) & (mid[:, 1] > 0) & (mid[:, 1] < mid[:, 0])    # wedge 0 < y < x removed
    return mesh.create_submesh(big, 2, np.where(~out)[0].astype(np.int32))[0]    # (mesh, entity map, vertex map, geom map)

def corner_exact(msh, omega):
    """u = r^lam sin(lam theta), lam = pi/omega, theta measured from one boundary ray of the corner (u = 0 on both rays)."""
    x = ufl.SpatialCoordinate(msh)
    r, th = ufl.sqrt(x[0] ** 2 + x[1] ** 2), ufl.atan2(x[1], x[0])       # atan2 in (-pi, pi]
    if omega < 1.6 * np.pi:
        th = ufl.conditional(ufl.lt(th, 0), th + 2 * ufl.pi, th)         # theta in [0, omega] for 5pi/4 and 3pi/2
    else:
        th = ufl.conditional(ufl.lt(th, np.pi / 4 - 1e-14), th + 2 * ufl.pi, th) - np.pi / 4   # [0, 7pi/4] from the ray y = x
    lam = np.pi / omega
    return r ** lam * ufl.sin(lam * th)

def corner_solve(msh, k, omega):
    """-Δu = 0 with u = exact on the boundary; returns (number of DOFs, L2 error, H1 error, u_h)."""
    V = fem.functionspace(msh, ("Lagrange", k))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    u_ex = corner_exact(msh, omega)
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    msh.topology.create_connectivity(1, 2)
    bc = fem.dirichletbc(g, fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology)))
    ndofs = V.dofmap.index_map.size_global
    uh = LinearProblem(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, fem.Constant(msh, 0.0) * v * ufl.dx, bcs=[bc],
                       petsc_options_prefix=f"corner_{k}_{ndofs}_", petsc_options={"ksp_type": "preonly", "pc_type": "lu"}).solve()
    return (ndofs, *errors(msh, uh - u_ex), uh)

def graded(H, mu, omega):
    """Refine the corner mesh (h = H) until h_K <= 1.5 H r^(1-mu), r = distance of the cell midpoint to the corner."""
    msh = corner_domain(int(round(1 / H)), omega)
    for _ in range(40):
        msh.topology.create_entities(1); msh.topology.create_connectivity(2, 1)
        cells = np.arange(msh.topology.index_map(2).size_local, dtype=np.int32)
        hK = msh.h(2, cells)
        r = np.linalg.norm(mesh.compute_midpoints(msh, 2, cells)[:, :2], axis=1)
        mark = cells[hK > 1.5 * H * np.maximum(r, H ** (1 / mu)) ** (1 - mu)]
        if msh.comm.allreduce(len(mark), op=MPI.SUM) == 0:                # collective: every rank decides together
            break
        edges = mesh.compute_incident_entities(msh.topology, mark, 2, 1)
        msh = mesh.refine(msh, edges)[0]                                   # refine returns (mesh, parent cells, parent facets)
    return msh

def dof_slopes(R):
    """Slopes of the errors against the number of DOFs: -d log E / d log N_dof (columns of R: N_dof, L2, H1)."""
    R = np.asarray(R, dtype=float)
    return [np.round(-np.diff(np.log(R[:, j])) / np.diff(np.log(R[:, 0])), 2) for j in (1, 2)]

out = {}
for label, meshes in (("uniform", [corner_domain(N, 1.5 * np.pi) for N in (4, 8, 16, 32, 64)]),
                      ("graded mu=0.5", [graded(H, 0.5, 1.5 * np.pi) for H in (1 / 4, 1 / 8, 1 / 16, 1 / 32)])):
    R = np.array([corner_solve(m, 1, 1.5 * np.pi)[:3] for m in meshes])
    out[label] = R
    s2, s1 = dof_slopes(R)
    if MPI.COMM_WORLD.rank == 0:
        print(f"P1 {label:14s} DOFs {R[:, 0].astype(int).tolist()}  H1 {[f'{e:.2e}' for e in R[:, 2]]}  "
              f"slopes vs DOFs: H1 {s1}  L2 {s2}   (optimal H1 slope 0.5)")
if MPI.COMM_WORLD.rank == 0:
    np.savez("/tmp/fnx/graded.npz", uniform=out["uniform"], graded=out["graded mu=0.5"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load("/tmp/fnx/graded.npz")
fig, ax = plt.subplots(figsize=(6.5, 4))
for key, style in (("uniform", "o--"), ("graded", "s-")):
    ax.loglog(d[key][:, 0], d[key][:, 2], style, label=f"P1, {key}")
n = np.array([1e2, 2e4])
ax.loglog(n, 0.9 * n ** (-1 / 3), "k:", lw=1); ax.text(3e3, 0.9 * 3e3 ** (-1 / 3) * 1.15, "$N^{-1/3}$")
ax.loglog(n, 1.2 * n ** (-1 / 2), "k-.", lw=1); ax.text(3e3, 1.2 * 3e3 ** (-1 / 2) * 0.7, "$N^{-1/2}$")
ax.set_xlabel("number of DOFs"); ax.set_ylabel("$|u-u_h|_{H^1}$"); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); plt.show()

> **✏️ Your turn (5 min).** P2 on graded meshes with $\mu=0.5$ and with $\mu=0.25$ ($H=\tfrac14,\dots,\tfrac1{16}$). Which $\mu$ satisfies $\mu<\lambda/k$?
> Compare the $H^1$ slopes against $N_{\rm dof}$ with the optimal value $k/2=1$.

In [ ]:
%%fenicsx -np 2
# TODO: P2, graded meshes with mu = 0.5 and mu = 0.25

## 5 · LLM-assisted coding: live demo  *(lecture Part D)*
Prompt (paste the API reference sheet first):

```text
Context: [API reference sheet]
Task: DOLFINx 0.11, %%fenicsx -np 2 cell. Convergence study for -Δu = 0 on the L-shaped domain [-1,1]^2 minus the quadrant x>0, y<0,
with u = r^(2/3) sin(2θ/3) on the boundary. Build the domain with mesh.create_submesh from a create_rectangle mesh (h = 1/N, N = 4, 8, 16, 32).
P2 elements. Before computing, state the expected H1 and L2 rates and why. Then report errors and rates on rank 0.
```
Paste the result below **unchanged**, run it, then go through the checklist.

In [ ]:
%%fenicsx -np 2
# TODO: paste the %%fenicsx part your LLM produced (unchanged) and run it

**Verification checklist** — checks with a known answer

- [ ] The LLM's **predicted** rates: $H^1$ $\tfrac23$, $L^2$ $\tfrac43$ — not 2 and 3 (P2 on a convex domain)
- [ ] $\theta$ is in $[0,\tfrac{3\pi}2]$ on the whole domain (`atan2` returns $(-\pi,\pi]$): $u_h\ge0$ everywhere
- [ ] `create_submesh` returns a tuple; the mesh is element `[0]`
- [ ] Errors integrated directly with a UFL exact solution and `allreduce`; rates stable over the last two refinements
- [ ] The numbers are identical with `-np 1` and `-np 2`, and printed once